<a href="https://www.kaggle.com/code/kiranraocsegpcet/cru-flood?scriptVersionId=372052522" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# CRU-Flood: Corrective Region Unlearning for Flood Segmentation (FloodNet)

This notebook runs every experiment in the article and saves the results to `/kaggle/working/cru_flood/results`
(CSV files and figures) and `/kaggle/working/cru_flood/cru_flood_results.zip`.

**Before running**
1. *Settings → Accelerator*: choose a GPU (T4 or P100). *Settings → Internet*: on (needed to install one library and download ImageNet weights).
2. *Add Input*: attach **aletbm/aerial-imagery-dataset-floodnet-challenge** (FloodNet Challenge Track 1, 398 images with 10-class masks).
   The notebook finds it anywhere under `/kaggle/input` and, because only the training masks are released, makes a fixed
   (seeded) 60/20/20 split: 238 train / 79 validation / 81 test. FloodNet-Supervised_v1.0 (2,343 images, official split) also works.
   A two-class flood/no-flood dataset (such as *Flood Area Segmentation*) will not work.
   Run the **Check the dataset** cell before the long run: it lists what it found.
3. Leave `SMOKE_TEST = False` in the Settings cell (the default) and use *Save Version → Save & Run All (Commit)*
   so the run continues in the background. Download `cru_flood_results.zip` from the Output tab when it finishes.
4. `SMOKE_TEST = True` is only a few-minute check that the code runs. Its numbers are **not** results, and its zip is named
   `SMOKE_TEST_not_results.zip` so the two cannot be confused.

**Long runs.** A full run takes several GPU hours. Kaggle sessions are time-limited, so the work is split into stages:
* Session 1: `STAGE = 'train'` trains the 8 base models.
* Session 2: add the output of session 1 as an input, set `RESUME_FROM` to its `cru_flood` folder, `STAGE = 'unlearn'`, `PARTS = ('main',)`.
* Session 3: same as session 2 with `PARTS = ('tau', 'ablations', 'idfrac', 'real')`.

Every finished run is recorded in `results/done.json`, so an interrupted session can be resumed the same way.
When everything has finished, download `cru_flood_results.zip` and send it back for the Results section.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('segmentation_models_pytorch') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch==0.6.0'], check=True)

## Code
The cell below writes the experiment code to `cruflood.py`.

In [ ]:
%%writefile cruflood.py
"""CRU-Flood: Corrective Region Unlearning for Flood segmentation on FloodNet.

Runs every experiment in the article and writes CSV results and figures:
  Stage 1  base models: clean and corrupted training masks, two architectures
  Stage 2  CRU-Flood, unlearning baselines, retraining without the forget set,
           threshold sensitivity, ablations, identification-fraction study,
           real-label setting

All numbers in the article's Results section must come from the files this
script writes in <work_dir>/results.
"""
import argparse
import copy
import dataclasses
import glob
import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from dataclasses import asdict, dataclass, field

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy import ndimage
from skimage.segmentation import slic

import segmentation_models_pytorch as smp

CLASSES = ['background', 'building-flooded', 'building-non-flooded', 'road-flooded',
           'road-non-flooded', 'water', 'tree', 'vehicle', 'pool', 'grass']
K = len(CLASSES)
PAIR_CLASSES = (1, 2, 3, 4)
PARTNER = np.arange(256, dtype=np.uint8)
PARTNER[1], PARTNER[2], PARTNER[3], PARTNER[4] = 2, 1, 4, 3
IS_PAIR = np.zeros(256, dtype=bool)
IS_PAIR[list(PAIR_CLASSES)] = True
IGNORE = 255
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
RUN_VERSION = '2026-11-16'                   # change when results from older code must not be reused


# --------------------------------------------------------------------------- config
@dataclass
class Config:
    data_root: str = ''                      # auto-detected under /kaggle/input when empty
    work_dir: str = '/kaggle/working/cru_flood'
    resume_from: str = ''                    # folder of an earlier run whose models/ and cache/ to reuse
    stage: str = 'all'                       # 'train', 'unlearn' or 'all'
    parts: tuple = ('main', 'tau', 'ablations', 'idfrac', 'real')
    width: int = 512
    height: int = 384
    models: tuple = ('deeplab', 'segformer')
    encoders: dict = field(default_factory=lambda: {'deeplab': 'resnet50', 'segformer': 'mit_b2'})
    pretrained: bool = True
    noise_rates: tuple = (0.1, 0.2, 0.3)
    main_eta: float = 0.2
    min_area: int = 64
    epochs: int = 30
    early_frac: float = 0.2
    batch_size: int = 8
    lr: float = 2e-4
    weight_decay: float = 1e-4
    num_workers: int = 2
    mc_passes: int = 10
    mc_p: float = 0.1
    weights: tuple = (1 / 3, 1 / 3, 1 / 3)
    tau: float = 0.5
    slic_segments: int = 400
    slic_compactness: float = 10.0
    lora_r: int = 8
    lora_alpha: float = 16.0
    unl_epochs: int = 5
    unl_epochs_later: int = 2              # epochs in rounds 2 and later
    unl_lr: float = 1e-3                     # adapters
    base_unl_lr: float = 1e-4                # baselines that update all weights
    lam_fgt: float = 1.0
    lam_kd: float = 1.0
    lam_ret: float = 0.5
    kd_T: float = 2.0
    rounds: int = 3
    stop_frac: float = 0.01
    uncertain_mode: str = 'auto'             # flagged regions without peer agreement: 'forget', 'ignore' or 'auto' (validation)
    tune: bool = True                        # choose baseline settings and uncertain_mode on validation mIoU
    tune_grid: dict = field(default_factory=lambda: {
        'cru_flood': [{'uncertain_mode': 'forget'}, {'uncertain_mode': 'ignore'}],
        'finetune': [{'base_unl_lr': 1e-5}, {'base_unl_lr': 1e-4}],
        'neggrad': [{'neggrad_beta': 0.01}, {'neggrad_beta': 0.1}],
        'scrub': [{'base_unl_lr': 1e-5, 'scrub_kl_cap': 0.5}, {'base_unl_lr': 1e-5, 'scrub_kl_cap': 2.0},
                  {'base_unl_lr': 1e-4, 'scrub_kl_cap': 0.5}, {'base_unl_lr': 1e-4, 'scrub_kl_cap': 2.0}],
        'ssd': [{'ssd_alpha': 10.0}, {'ssd_alpha': 50.0}, {'ssd_alpha': 100.0}]})
    seeds: tuple = (0, 1, 2)
    neggrad_beta: float = 0.1
    neggrad_clip: float = 10.0
    scrub_kl_cap: float = 2.0                # SCRUB max-steps stop pushing once the mean KL reaches this
    ssd_alpha: float = 10.0
    ssd_lambda: float = 1.0
    run_rewod: bool = True
    tau_grid: tuple = (0.3, 0.4, 0.5, 0.6, 0.7)
    id_fractions: tuple = (0.1, 0.25, 0.5, 1.0)
    ablation_model: str = 'segformer'
    ece_bins: int = 15
    eval_stride: int = 4
    amp: bool = True
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    smoke: bool = False                      # True only for the quick check; its numbers are not results


def tiny_config(**kw):
    """Small settings for a CPU smoke test on synthetic data; not for results."""
    cfg = Config(width=128, height=96, encoders={'deeplab': 'resnet18', 'segformer': 'mit_b0'},
                 pretrained=False, epochs=3, batch_size=4, num_workers=0, mc_passes=2,
                 slic_segments=60, unl_epochs=1, rounds=2, seeds=(0, 1), amp=False,
                 tau_grid=(0.4, 0.5), id_fractions=(0.5, 1.0), min_area=16, smoke=True)
    for k, v in kw.items():
        setattr(cfg, k, v)
    return cfg


# --------------------------------------------------------------------------- utils
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def log(*a):
    print(time.strftime('%H:%M:%S'), *a, flush=True)


def append_row(path, row):
    """Add one row; rows may carry different columns, so the file is rewritten whole."""
    df = pd.DataFrame([row])
    if os.path.exists(path):
        df = pd.concat([pd.read_csv(path), df], ignore_index=True)
    df.to_csv(path, index=False)


class Done:
    """Keys of finished runs, so an interrupted Kaggle session can resume."""

    def __init__(self, path):
        self.path = path
        self.keys = set(json.load(open(path))) if os.path.exists(path) else set()

    def __contains__(self, k):
        return k in self.keys

    def add(self, k):
        self.keys.add(k)
        json.dump(sorted(self.keys), open(self.path, 'w'))


def autocast(cfg):
    return torch.autocast(device_type='cuda', dtype=torch.float16,
                          enabled=cfg.amp and cfg.device == 'cuda')


# --------------------------------------------------------------------------- data
IMG_EXT = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')
SPLIT_RE = {'train': re.compile(r'^(train|training)([-_ ].*)?$'),
            'val': re.compile(r'^(val|valid|validation)([-_ ].*)?$'),
            'test': re.compile(r'^(test|testing)([-_ ].*)?$')}
LABEL_DIR_RE = re.compile(r'(label(?!ed)|mask|annot|(^|[-_ ])gt($|[-_ ]))')
LABEL_SUFFIX_RE = re.compile(r'[-_](lab|label|mask|gt)$', re.I)
FLOODNET_RGB = {tuple(int(v) for v in c): i for i, c in enumerate(
    [[0, 0, 0], [255, 0, 0], [180, 120, 120], [160, 150, 20], [140, 140, 140],
     [61, 230, 250], [0, 82, 255], [255, 0, 245], [255, 235, 0], [4, 250, 7]])}


def _split_of(path, root):
    for part in reversed(os.path.relpath(path, root).lower().split(os.sep)):
        for split, rx in SPLIT_RE.items():
            if rx.match(part):
                return split
    return None


def describe_inputs(root='/kaggle/input', max_dirs=40):
    """Folders holding image files, with counts and sample names, to diagnose a missing dataset."""
    lines = []
    if not os.path.isdir(root):
        return f'{root} does not exist.'
    for d, _, files in sorted(os.walk(root)):
        imgs = sorted(f for f in files if f.lower().endswith(IMG_EXT))
        if imgs:
            lines.append(f'{d}  ({len(imgs)} images, e.g. {", ".join(imgs[:2])})')
        if len(lines) >= max_dirs:
            lines.append('...')
            break
    if not lines:
        top = sorted(os.listdir(root))
        return f'No image files under {root}. Top-level entries: {top if top else "none (no dataset attached)"}'
    return '\n'.join(lines)


def find_floodnet(root_hint=''):
    """Return {split: [(image_path, label_path), ...]}.

    Accepts the official FloodNet-Supervised_v1.0 layout (train/train-org-img, train/train-label-img, ...)
    and common re-uploads (images/masks folders, '_lab' or '_mask' suffixes, val/valid/validation),
    nested at any depth under the root.
    """
    root = root_hint or '/kaggle/input'
    if not os.path.isdir(root):
        raise FileNotFoundError(f'{root} does not exist. Set DATA_ROOT to the FloodNet folder.')
    found = {s: {} for s in SPLIT_RE}
    for d, _, files in os.walk(root):
        dir_name = os.path.basename(d).lower()
        split = _split_of(d, root)
        if split is None or '__macosx' in d.lower():
            continue
        label_dir = bool(LABEL_DIR_RE.search(dir_name)) and 'color' not in dir_name
        color_dir = 'color' in dir_name
        for f in files:
            if not f.lower().endswith(IMG_EXT) or f.startswith('._'):
                continue
            stem = os.path.splitext(f)[0]
            is_label = label_dir or color_dir or bool(LABEL_SUFFIX_RE.search(stem))
            key = LABEL_SUFFIX_RE.sub('', stem)
            entry = found[split].setdefault(key, {})
            path = os.path.join(d, f)
            if is_label:
                # prefer class-index masks over colour masks when both exist
                if 'lab' not in entry or ('color' in entry['lab'].lower() and not color_dir):
                    entry['lab'] = path
            elif 'img' not in entry:
                entry['img'] = path
    pairs = {s: sorted((e['img'], e['lab']) for e in v.values() if 'img' in e and 'lab' in e)
             for s, v in found.items()}
    counts = {s: len(v) for s, v in pairs.items()}
    if counts['train'] and counts['val'] and counts['test']:
        log(f'FloodNet found under {root}: {counts}')
        return pairs
    pool = sorted(set(pairs['train'] + pairs['val'] + pairs['test']))
    if len(pool) >= 30:
        rng = np.random.default_rng(0)
        order = rng.permutation(len(pool))
        n_tr, n_va = int(0.6 * len(pool)), int(0.2 * len(pool))
        log(f'WARNING: labelled pairs found only for {counts}. Using a seeded 60/20/20 split of '
            f'{len(pool)} labelled images instead of the official split; the article must say so.')
        return {'train': [pool[i] for i in order[:n_tr]], 'val': [pool[i] for i in order[n_tr:n_tr + n_va]],
                'test': [pool[i] for i in order[n_tr + n_va:]], 'fallback_split': True}
    raise FileNotFoundError(
        f'FloodNet image/mask pairs not found under {root} (pairs per split: {counts}).\n'
        'Folders with images:\n' + describe_inputs(root) +
        '\nAttach the FloodNet-Supervised_v1.0 dataset, or set DATA_ROOT to the folder holding train/, val/ and test/.')


def _to_index_mask(lb):
    if lb.ndim == 2:
        return lb
    lb = lb[..., :3]
    if (lb[..., 0] == lb[..., 1]).all() and (lb[..., 1] == lb[..., 2]).all():
        return lb[..., 0]
    out = np.full(lb.shape[:2], IGNORE, dtype=np.uint8)
    for rgb, idx in FLOODNET_RGB.items():
        out[(lb == rgb).all(-1)] = idx
    return out


def _load_pair(args):
    ip, lp, w, h = args
    im = Image.open(ip)
    im.draft('RGB', (w, h))
    im = np.asarray(im.convert('RGB').resize((w, h), Image.BILINEAR), dtype=np.uint8)
    lb = Image.open(lp)
    if lb.mode not in ('L', 'P', 'I', 'I;16', 'RGB', 'RGBA'):
        lb = lb.convert('RGB')
    lb = np.asarray(lb.resize((w, h), Image.NEAREST))
    return im, _to_index_mask(lb).astype(np.uint8)


def load_floodnet(cfg):
    cache_dir = os.path.join(cfg.work_dir, 'cache')
    os.makedirs(cache_dir, exist_ok=True)
    data, files = {}, None
    for s in ('train', 'val', 'test'):
        path = os.path.join(cache_dir, f'{s}_{cfg.width}x{cfg.height}.npz')
        if not os.path.exists(path):
            files = files or find_floodnet(cfg.data_root)
            pairs = files[s]
            log(f'Loading {len(pairs)} {s} images')
            with ThreadPoolExecutor(8) as ex:
                res = list(ex.map(_load_pair, [(a, b, cfg.width, cfg.height) for a, b in pairs]))
            imgs = np.stack([r[0] for r in res])
            masks = np.stack([r[1] for r in res])
            names = np.array([os.path.basename(a) for a, _ in pairs])
            np.savez(path, imgs=imgs, masks=masks, names=names, fallback=bool(files.get('fallback_split', False)))
        z = np.load(path)
        data[s] = {'imgs': z['imgs'], 'masks': z['masks'], 'names': z['names']}
        vals = np.unique(data[s]['masks'])
        known = vals[vals != IGNORE]
        if set(known.tolist()) <= {0, 1} or set(vals.tolist()) <= {0, 255}:
            raise ValueError(f'The {s} masks hold only {vals.tolist()}: this looks like a binary flood/no-flood '
                             'dataset, not FloodNet with its 10 classes.')
        if known.max() >= K:
            raise ValueError(f'Unexpected label values in the {s} masks: {vals.tolist()} (FloodNet uses 0-9).')
        log(f'{s}: {len(data[s]["imgs"])} images, label values {vals.tolist()}')
    zt = np.load(os.path.join(cache_dir, f'train_{cfg.width}x{cfg.height}.npz'))
    data['fallback_split'] = bool(zt['fallback']) if 'fallback' in zt.files else False
    return data


class ArrDS(torch.utils.data.Dataset):
    """Images plus any number of per-pixel arrays (masks, roles, targets)."""

    def __init__(self, imgs, arrays, idx, augment=False, jitter=False):
        self.imgs, self.arrays, self.idx = imgs, arrays, np.asarray(idx)
        self.augment, self.jitter = augment, jitter

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, j):
        i = int(self.idx[j])
        x = self.imgs[i].astype(np.float32) / 255.0
        arrs = [a[i] for a in self.arrays]
        if self.augment:
            if random.random() < 0.5:
                x, arrs = x[:, ::-1], [a[:, ::-1] for a in arrs]
            if random.random() < 0.5:
                x, arrs = x[::-1], [a[::-1] for a in arrs]
        if self.jitter:
            x = np.clip(x * random.uniform(0.9, 1.1) + random.uniform(-0.05, 0.05), 0, 1)
        x = (x - MEAN) / STD
        out = [torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))]
        out += [torch.from_numpy(np.ascontiguousarray(a).astype(np.int64)) for a in arrs]
        return (*out, i)


def loader(cfg, imgs, arrays, idx, shuffle=False, augment=False, jitter=False, seed=0):
    g = torch.Generator()
    g.manual_seed(seed)
    return torch.utils.data.DataLoader(ArrDS(imgs, arrays, idx, augment, jitter), batch_size=cfg.batch_size,
                                       shuffle=shuffle, num_workers=cfg.num_workers, generator=g,
                                       pin_memory=cfg.device == 'cuda')


# --------------------------------------------------------------------------- corruption
def corrupt(masks, eta, seed, min_area):
    """Swap a fraction eta of flood-pair connected components to their partner class."""
    rng = np.random.default_rng(seed)
    struct = np.ones((3, 3), dtype=bool)
    comps = []
    for n in range(len(masks)):
        for c in PAIR_CLASSES:
            lab, k = ndimage.label(masks[n] == c, structure=struct)
            if k == 0:
                continue
            sizes = np.bincount(lab.ravel())[1:]
            for lid in np.nonzero(sizes >= min_area)[0] + 1:
                comps.append((n, c, int(lid)))
    chosen = rng.choice(len(comps), int(round(eta * len(comps))), replace=False) if comps else []
    by_img = {}
    for j in chosen:
        n, c, lid = comps[j]
        by_img.setdefault((n, c), []).append(lid)
    noisy = masks.copy()
    comp_map = np.zeros(masks.shape, dtype=np.int32)       # 1-based id of each swapped component
    cid = 0
    for (n, c), lids in sorted(by_img.items()):
        lab, _ = ndimage.label(masks[n] == c, structure=struct)
        for lid in lids:
            cid += 1
            m = lab == lid
            noisy[n][m] = PARTNER[c]
            comp_map[n][m] = cid
    C = comp_map > 0
    pair_px = IS_PAIR[masks].sum()
    stats = {'eta': eta, 'eligible_components': len(comps), 'swapped_components': cid,
             'swapped_pixels': int(C.sum()), 'pct_train_pixels': 100 * C.sum() / masks.size,
             'pct_flood_pair_pixels': 100 * C.sum() / max(pair_px, 1)}
    return noisy, C, comp_map, stats


# --------------------------------------------------------------------------- models
class MCDropout(nn.Dropout2d):
    """Dropout before the segmentation head; switched on alone for MC passes."""


def build_model(cfg, name):
    enc = cfg.encoders[name]
    w = 'imagenet' if cfg.pretrained else None
    if name == 'deeplab':
        m = smp.DeepLabV3Plus(encoder_name=enc, encoder_weights=w, classes=K)
    elif name == 'segformer':
        m = smp.Segformer(encoder_name=enc, encoder_weights=w, classes=K)
    else:
        raise ValueError(name)
    m.segmentation_head = nn.Sequential(MCDropout(cfg.mc_p), m.segmentation_head)
    return m


def set_mc(model, on):
    model.eval()
    for mod in model.modules():
        if isinstance(mod, MCDropout):
            mod.train(on)


def n_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_model(cfg, name, data, masks, tag, seed=0):
    """Train one base model; keep the early snapshot and the final weights."""
    out = os.path.join(cfg.work_dir, 'models', tag)
    final_p, early_p = os.path.join(out, 'final.pt'), os.path.join(out, 'early.pt')
    if os.path.exists(final_p) and os.path.exists(early_p):
        return final_p, early_p
    os.makedirs(out, exist_ok=True)
    set_seed(seed)
    dev = cfg.device
    model = build_model(cfg, name).to(dev)
    tr = data['train']
    dl = loader(cfg, tr['imgs'], [masks], np.arange(len(masks)), shuffle=True, augment=True, jitter=True, seed=seed)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    total = cfg.epochs * len(dl)
    warm = len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler('cuda', enabled=cfg.amp and dev == 'cuda')
    e_early = max(1, round(cfg.early_frac * cfg.epochs))
    hist = []
    t0 = time.time()
    for ep in range(1, cfg.epochs + 1):
        model.train()
        tot, nb = 0.0, 0
        for x, y, _ in dl:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with autocast(cfg):
                loss = F.cross_entropy(model(x), y, ignore_index=IGNORE)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item()
            nb += 1
        row = {'tag': tag, 'epoch': ep, 'train_loss': tot / max(nb, 1)}
        if ep == e_early:
            torch.save(model.state_dict(), early_p)
        if ep % 5 == 0 or ep == cfg.epochs or ep == e_early:
            row['val_miou'] = evaluate(cfg, model, data['val']['imgs'], data['val']['masks'])['miou']
        hist.append(row)
        log(f'[{tag}] epoch {ep}/{cfg.epochs} loss {row["train_loss"]:.4f}'
            + (f' val mIoU {row["val_miou"]:.4f}' if 'val_miou' in row else ''))
    torch.save(model.state_dict(), final_p)
    pd.DataFrame(hist).to_csv(os.path.join(out, 'history.csv'), index=False)
    json.dump({'train_seconds': time.time() - t0, 'early_epoch': e_early}, open(os.path.join(out, 'meta.json'), 'w'))
    return final_p, early_p


def load_model(cfg, name, path):
    m = build_model(cfg, name)
    m.load_state_dict(torch.load(path, map_location='cpu'))
    return m.to(cfg.device).eval()


# --------------------------------------------------------------------------- evaluation
@torch.no_grad()
def evaluate(cfg, model, imgs, masks):
    model.eval()
    dev, B = cfg.device, cfg.ece_bins
    conf = torch.zeros(K * K, dtype=torch.float64, device=dev)
    cnt = torch.zeros(B, dtype=torch.float64, device=dev)
    csum, asum = torch.zeros_like(cnt), torch.zeros_like(cnt)
    for x, y, _ in loader(cfg, imgs, [masks], np.arange(len(imgs))):
        x, y = x.to(dev), y.to(dev)
        with autocast(cfg):
            logits = model(x)
        p = logits.float().softmax(1)
        c, pred = p.max(1)
        v = y != IGNORE
        conf += torch.bincount(y[v] * K + pred[v], minlength=K * K).double()
        s = cfg.eval_stride
        cs, ps, ys = c[:, ::s, ::s], pred[:, ::s, ::s], y[:, ::s, ::s]
        vs = ys != IGNORE
        b = (cs[vs] * B).long().clamp(max=B - 1)
        cnt += torch.bincount(b, minlength=B).double()
        csum += torch.bincount(b, weights=cs[vs].double(), minlength=B)
        asum += torch.bincount(b, weights=(ps[vs] == ys[vs]).double(), minlength=B)
    cm = conf.view(K, K).cpu().numpy()
    tp = np.diag(cm)
    denom = cm.sum(0) + cm.sum(1) - tp
    iou = np.where(denom > 0, tp / np.maximum(denom, 1), np.nan)
    cnt, csum, asum = cnt.cpu().numpy(), csum.cpu().numpy(), asum.cpu().numpy()
    ece = float(np.sum(np.abs(asum - csum)) / max(cnt.sum(), 1))
    out = {'miou': float(np.nanmean(iou)), 'pair_miou': float(np.nanmean(iou[1:5])),
           'pixel_acc': float(tp.sum() / max(cm.sum(), 1)), 'ece': ece}
    out.update({f'iou_{c}': float(iou[c]) for c in range(K)})
    return out


@torch.no_grad()
def train_set_metrics(cfg, model, imgs, clean, noisy, C):
    """Correction and memorisation on swapped pixels; accuracy on all other pixels."""
    model.eval()
    dev = cfg.device
    corr = mem = n_c = ret = n_r = 0
    for x, yc, yn, cc, idx in loader(cfg, imgs, [clean, noisy, C.astype(np.uint8)], np.arange(len(imgs))):
        x = x.to(dev)
        with autocast(cfg):
            pred = model(x).argmax(1).cpu()
        cc = cc.bool()
        corr += (pred[cc] == yc[cc]).sum().item()
        mem += (pred[cc] == yn[cc]).sum().item()
        n_c += cc.sum().item()
        r = (~cc) & (yc != IGNORE)
        ret += (pred[r] == yc[r]).sum().item()
        n_r += r.sum().item()
    return {'correction_rate': corr / max(n_c, 1), 'memorisation_rate': mem / max(n_c, 1),
            'retain_acc': ret / max(n_r, 1)}


# --------------------------------------------------------------------------- selection signals
@torch.no_grad()
def predict_labels(cfg, model, imgs):
    model.eval()
    out = np.zeros(imgs.shape[:3], dtype=np.uint8)
    for x, idx in loader(cfg, imgs, [], np.arange(len(imgs))):
        with autocast(cfg):
            out[idx.numpy()] = model(x.to(cfg.device)).argmax(1).cpu().numpy().astype(np.uint8)
    return out


@torch.no_grad()
def compute_signals(cfg, model, imgs, given, seed=0):
    """Per-pixel loss of the given label, MC-dropout entropy and argmax of the mean prediction."""
    set_seed(seed)
    N = len(imgs)
    loss = np.zeros(given.shape, dtype=np.float16)
    unc = np.zeros(given.shape, dtype=np.float16)
    pf = np.zeros(given.shape, dtype=np.uint8)
    M = max(1, cfg.mc_passes)
    for x, y, idx in loader(cfg, imgs, [given], np.arange(N)):
        x, y = x.to(cfg.device), y.to(cfg.device)
        set_mc(model, M > 1)
        p = 0
        for _ in range(M):
            with autocast(cfg):
                p = p + model(x).float().softmax(1)
        p = p / M
        set_mc(model, False)
        py = p.gather(1, y.clamp(max=K - 1)[:, None])[:, 0]
        i = idx.numpy()
        loss[i] = (-torch.log(py + 1e-8)).cpu().numpy().astype(np.float16)
        unc[i] = (-(p * torch.log(p + 1e-8)).sum(1) / math.log(K)).cpu().numpy().astype(np.float16)
        pf[i] = p.argmax(1).cpu().numpy().astype(np.uint8)
    rank = np.zeros(given.shape, dtype=np.float16)
    rng = np.random.default_rng(seed)
    for c in PAIR_CLASSES:
        m = given == c
        vals = loss[m].astype(np.float32)
        if vals.size == 0:
            continue
        sub = np.sort(rng.choice(vals, min(2_000_000, vals.size), replace=False))
        rank[m] = (np.searchsorted(sub, vals, side='right') / sub.size).astype(np.float16)
    return {'rank': rank, 'unc': unc, 'pf': pf}


def _slic_one(args):
    img, n_seg, comp = args
    return slic(img, n_segments=n_seg, compactness=comp, start_label=0, channel_axis=-1).astype(np.uint16)


def compute_slic(cfg, imgs):
    path = os.path.join(cfg.work_dir, 'cache', f'slic_{cfg.width}x{cfg.height}_{cfg.slic_segments}.npy')
    if os.path.exists(path):
        return np.load(path)
    log(f'SLIC superpixels for {len(imgs)} images')
    args = [(im, cfg.slic_segments, cfg.slic_compactness) for im in imgs]
    workers = max(1, os.cpu_count() or 1)
    if workers > 1:
        with ProcessPoolExecutor(workers) as ex:
            segs = np.stack(list(ex.map(_slic_one, args, chunksize=8)))
    else:
        segs = np.stack([_slic_one(a) for a in args])
    np.save(path, segs)
    return segs


def select_regions(given, segs, sig, pg, w=(1 / 3, 1 / 3, 1 / 3), tau=0.5, budget=None,
                   pixel_level=False, forget_only=False):
    """Flag flood-pair regions; return role (0 keep, 1 correct, 2 forget) and correct targets."""
    N = len(given)
    per = []
    for n in range(N):
        g = given[n]
        pm = IS_PAIR[g]
        if not pm.any():
            per.append(None)
            continue
        pf, gp = sig['pf'][n], pg[n]
        agree = (pf == gp) & (pf != g)
        s = (w[0] * sig['rank'][n].astype(np.float32) + w[1] * sig['unc'][n].astype(np.float32)
             + w[2] * agree.astype(np.float32))[pm]
        agp = ((pf == gp) & (pf == PARTNER[g]))[pm].astype(np.float32)
        if pixel_level:
            inv = np.arange(s.size)
            cnt = np.ones(s.size)
            S, frac = s, agp
        else:
            rid = (segs[n].astype(np.int64) * K + g)[pm]
            _, inv = np.unique(rid, return_inverse=True)
            cnt = np.bincount(inv)
            S = np.bincount(inv, s) / cnt
            frac = np.bincount(inv, agp) / cnt
        per.append((pm, inv, cnt, S, frac))
    tau_eff = tau
    if budget is not None:
        allS = np.concatenate([p[3] for p in per if p is not None])
        allC = np.concatenate([p[2] for p in per if p is not None])
        order = np.argsort(-allS)
        cum = np.cumsum(allC[order])
        k = int(np.searchsorted(cum, budget))
        tau_eff = float(allS[order[min(k, len(order) - 1)]])
    role = np.zeros(given.shape, dtype=np.uint8)
    target = np.full(given.shape, IGNORE, dtype=np.uint8)
    n_reg = 0
    for n, p in enumerate(per):
        if p is None:
            continue
        pm, inv, cnt, S, frac = p
        flag = S >= tau_eff
        corr = flag & (frac >= 0.5) & (not forget_only)
        n_reg += int(flag.sum())
        pf_, pc_ = flag[inv], corr[inv]
        rn, tn = role[n], target[n]
        rn[pm] = np.where(pc_, 1, np.where(pf_, 2, 0)).astype(np.uint8)
        tn[pm] = np.where(pc_, PARTNER[given[n][pm]], IGNORE).astype(np.uint8)
    return role, target, {'tau_eff': tau_eff, 'flagged_regions': n_reg}


def selection_stats(role, C=None, given=None, segs=None):
    flagged = role > 0
    st = {'flagged_px': int(flagged.sum()), 'correct_px': int((role == 1).sum()),
          'forget_px': int((role == 2).sum())}
    if C is not None:
        tp = int((flagged & C).sum())
        st['precision'] = tp / max(st['flagged_px'], 1)
        st['recall'] = tp / max(int(C.sum()), 1)
        st['f1'] = 2 * st['precision'] * st['recall'] / max(st['precision'] + st['recall'], 1e-12)
        st['correct_on_swapped_px'] = int(((role == 1) & C).sum())
        st['correct_on_clean_px'] = int(((role == 1) & ~C).sum())
        st['forget_on_swapped_px'] = int(((role == 2) & C).sum())
        st['correct_precision'] = st['correct_on_swapped_px'] / max(st['correct_px'], 1)
        st['forget_precision'] = st['forget_on_swapped_px'] / max(st['forget_px'], 1)
        st['random_precision'] = int(C.sum()) / max(int(IS_PAIR[given].sum()), 1) if given is not None else float('nan')
    return st


# --------------------------------------------------------------------------- LoRA
class LoRAConv(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale, self.enabled = base, alpha / r, True
        self.A = nn.Conv2d(base.in_channels, r, base.kernel_size, base.stride, base.padding, base.dilation, bias=False)
        self.B = nn.Conv2d(r, base.out_channels, 1, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * self.B(self.A(x)) if self.enabled else y

    def merged(self):
        delta = torch.einsum('or,rikl->oikl', self.B.weight[:, :, 0, 0], self.A.weight) * self.scale
        self.base.weight.data += delta.to(self.base.weight.dtype)
        return self.base


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale, self.enabled = base, alpha / r, True
        self.A = nn.Linear(base.in_features, r, bias=False)
        self.B = nn.Linear(r, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * self.B(self.A(x)) if self.enabled else y

    def merged(self):
        self.base.weight.data += (self.B.weight @ self.A.weight * self.scale).to(self.base.weight.dtype)
        return self.base


def apply_lora(model, r, alpha):
    for p in model.parameters():
        p.requires_grad = False

    def wrap(mod):
        for name, ch in mod.named_children():
            if isinstance(ch, nn.Conv2d) and ch.groups == 1:
                setattr(mod, name, LoRAConv(ch, r, alpha))
            elif isinstance(ch, nn.Linear):
                setattr(mod, name, LoRALinear(ch, r, alpha))
            elif not isinstance(ch, (LoRAConv, LoRALinear)):
                wrap(ch)

    wrap(model.decoder)
    wrap(model.segmentation_head)
    model.to(next(model.parameters()).device)
    return model


def set_lora(model, on):
    for m in model.modules():
        if isinstance(m, (LoRAConv, LoRALinear)):
            m.enabled = on


def merge_lora(model):
    def walk(mod):
        for name, ch in mod.named_children():
            if isinstance(ch, (LoRAConv, LoRALinear)):
                setattr(mod, name, ch.merged())
            else:
                walk(ch)
    walk(model)
    for p in model.parameters():
        p.requires_grad = False
    return model


# --------------------------------------------------------------------------- unlearning losses and loops
def kd_term(logits, t_logits, T):
    return (F.softmax(t_logits / T, 1) * (F.log_softmax(t_logits / T, 1) - F.log_softmax(logits / T, 1))).sum(1) * T * T


def cru_loss(cfg, logits, t_logits, y, role, target, lam_fgt, lam_kd, lam_ret):
    logits = logits.float()
    logp = F.log_softmax(logits, 1)
    corr, fgt = role == 1, role == 2          # role 3: flagged but ignored, no loss at all
    ret = (role == 0) & (y != IGNORE)
    L = logits.new_zeros(())
    if corr.any():
        L = L + F.nll_loss(logp, torch.where(corr, target, torch.full_like(target, IGNORE)), ignore_index=IGNORE)
    if fgt.any() and lam_fgt > 0:
        pgv = logp.gather(1, y.clamp(max=K - 1)[:, None])[:, 0].exp().clamp(max=1 - 1e-6)
        L = L + lam_fgt * (-torch.log1p(-pgv))[fgt].mean()
    if ret.any():
        if lam_kd > 0:
            L = L + lam_kd * kd_term(logits, t_logits.float(), cfg.kd_T)[ret].mean()
        if lam_ret > 0:
            L = L + lam_ret * F.nll_loss(logp, torch.where(ret, y, torch.full_like(y, IGNORE)), ignore_index=IGNORE)
    return L


def unl_indices(role, seed):
    rng = np.random.default_rng(seed)
    has = role.reshape(len(role), -1).max(1) > 0
    flagged, others = np.nonzero(has)[0], np.nonzero(~has)[0]
    extra = rng.choice(others, min(len(flagged), len(others)), replace=False) if len(others) else others
    return np.concatenate([flagged, extra])


def train_cru(cfg, model, imgs, given, role, target, seed, lam_fgt=None, lam_kd=None, lam_ret=None, teacher=None,
              epochs=None):
    """Train adapters (or all weights when teacher is given) with the CRU-Flood objective."""
    lam_fgt = cfg.lam_fgt if lam_fgt is None else lam_fgt
    lam_kd = cfg.lam_kd if lam_kd is None else lam_kd
    lam_ret = cfg.lam_ret if lam_ret is None else lam_ret
    dev = cfg.device
    idx = unl_indices(role, seed)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=cfg.unl_lr if teacher is None else cfg.base_unl_lr)
    scaler = torch.amp.GradScaler('cuda', enabled=cfg.amp and dev == 'cuda')
    for ep in range(cfg.unl_epochs if epochs is None else epochs):
        model.eval()
        for x, y, r, t, _ in loader(cfg, imgs, [given, role, target], idx, shuffle=True, augment=True, seed=seed + ep):
            x, y, r, t = x.to(dev), y.to(dev), r.to(dev), t.to(dev)
            with torch.no_grad(), autocast(cfg):
                if teacher is None:
                    set_lora(model, False)
                    t_logits = model(x)
                    set_lora(model, True)
                else:
                    t_logits = teacher(x)
            with autocast(cfg):
                logits = model(x)
            loss = cru_loss(cfg, logits, t_logits, y, r, t, lam_fgt, lam_kd, lam_ret)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()


def run_cru(cfg, f_final, f_early, peer_labels, imgs, given, segs, seed, C=None, sig1=None,
            rounds=None, w=None, early=True, pixel_level=False, forget_only=False,
            full_ft=False, lam_kd=None, budget=None):
    """CRU-Flood: progressive selection, then selective unlearning with retention."""
    rounds = cfg.rounds if rounds is None else rounds
    w = cfg.weights if w is None else w
    set_seed(seed)
    model = copy.deepcopy(f_final)
    teacher = None
    if full_ft:
        teacher = copy.deepcopy(f_final).eval()
        for p in teacher.parameters():
            p.requires_grad = False
        for p in model.parameters():
            p.requires_grad = True
    else:
        apply_lora(model, cfg.lora_r, cfg.lora_alpha)
    trainable = n_trainable(model)
    role_cum = np.zeros(given.shape, dtype=np.uint8)
    tgt_cum = np.full(given.shape, IGNORE, dtype=np.uint8)
    logs, n1, first_role = [], None, None
    t0 = time.time()
    for k in range(1, rounds + 1):
        if k == 1:
            sig = sig1 if (sig1 is not None and early) else compute_signals(cfg, f_early if early else f_final, imgs, given, seed)
        else:
            sig = compute_signals(cfg, model, imgs, given, seed + k)
        role, target, info = select_regions(given, segs, sig, peer_labels, w, cfg.tau, budget if k == 1 else None,
                                            pixel_level, forget_only)
        new = (role > 0) & (role_cum == 0)
        n_new = int(new.sum())
        if k == 1:
            n1, first_role = max(n_new, 1), role.copy()
        elif n_new < cfg.stop_frac * n1:
            logs.append({'round': k, 'new_px': n_new, 'stopped': True, **info})
            break
        role_cum[new], tgt_cum[new] = role[new], target[new]
        rec = {'round': k, 'new_px': n_new, 'stopped': False, 'uncertain_mode': cfg.uncertain_mode, **info,
               **selection_stats(role_cum, C, given)}
        logs.append(rec)
        log(f'  round {k}: {n_new} new flagged px' + (f', P {rec["precision"]:.3f} R {rec["recall"]:.3f}' if C is not None else ''))
        if n_new:
            role_train = np.where(role_cum == 2, 3, role_cum).astype(np.uint8) if cfg.uncertain_mode == 'ignore' else role_cum
            train_cru(cfg, model, imgs, given, role_train, tgt_cum, seed + 100 * k, lam_kd=lam_kd, teacher=teacher,
                      epochs=cfg.unl_epochs if k == 1 else cfg.unl_epochs_later)
    if not full_ft:
        merge_lora(model)
    model.eval()
    return model, {'logs': logs, 'seconds': time.time() - t0, 'trainable': trainable, 'first_role': first_role,
                   'role': role_cum}


def full_copy(model):
    m = copy.deepcopy(model)
    for p in m.parameters():
        p.requires_grad = True
    return m.eval()


def make_step(cfg, opt):
    scaler = torch.amp.GradScaler('cuda', enabled=cfg.amp and cfg.device == 'cuda')

    def step(loss):
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    return step


def run_finetune(cfg, f0, imgs, given, forget, seed):
    model = full_copy(f0)
    role = forget.astype(np.uint8) * 2
    step = make_step(cfg, torch.optim.Adam(model.parameters(), lr=cfg.base_unl_lr))
    idx = unl_indices(role, seed)
    for ep in range(cfg.unl_epochs):
        for x, y, r, _ in loader(cfg, imgs, [given, role], idx, shuffle=True, augment=True, seed=seed + ep):
            x, y, r = x.to(cfg.device), y.to(cfg.device), r.to(cfg.device)
            with autocast(cfg):
                logits = model(x)
            yr = torch.where(r == 0, y, torch.full_like(y, IGNORE))
            if (yr != IGNORE).any():
                step(F.cross_entropy(logits.float(), yr, ignore_index=IGNORE))
    return model.eval()


def run_neggrad(cfg, f0, imgs, given, forget, seed):
    model = full_copy(f0)
    role = forget.astype(np.uint8) * 2
    step = make_step(cfg, torch.optim.Adam(model.parameters(), lr=cfg.base_unl_lr))
    idx = unl_indices(role, seed)
    for ep in range(cfg.unl_epochs):
        for x, y, r, _ in loader(cfg, imgs, [given, role], idx, shuffle=True, augment=True, seed=seed + ep):
            x, y, r = x.to(cfg.device), y.to(cfg.device), r.to(cfg.device)
            with autocast(cfg):
                logits = model(x).float()
            yr = torch.where(r == 0, y, torch.full_like(y, IGNORE))
            yf = torch.where(r > 0, y, torch.full_like(y, IGNORE))
            loss = logits.new_zeros(())
            if (yr != IGNORE).any():
                loss = loss + F.cross_entropy(logits, yr, ignore_index=IGNORE)
            if (yf != IGNORE).any():
                loss = loss - cfg.neggrad_beta * F.cross_entropy(logits, yf, ignore_index=IGNORE).clamp(max=cfg.neggrad_clip)
            if loss.requires_grad:
                step(loss)
    return model.eval()


def run_scrub(cfg, f0, imgs, given, forget, seed):
    student, teacher = full_copy(f0), copy.deepcopy(f0).eval()
    for p in teacher.parameters():
        p.requires_grad = False
    role = forget.astype(np.uint8) * 2
    step = make_step(cfg, torch.optim.Adam(student.parameters(), lr=cfg.base_unl_lr))
    idx = unl_indices(role, seed)
    fidx = np.nonzero(role.reshape(len(role), -1).max(1) > 0)[0]
    msteps = max(1, cfg.unl_epochs // 2)
    T = cfg.kd_T
    for ep in range(cfg.unl_epochs):
        if ep < msteps:
            for x, y, r, _ in loader(cfg, imgs, [given, role], fidx, shuffle=True, augment=True, seed=seed + ep):
                x, r = x.to(cfg.device), r.to(cfg.device)
                with torch.no_grad(), autocast(cfg):
                    t = teacher(x)
                with autocast(cfg):
                    s = student(x)
                m = r > 0
                if m.any():
                    step(-kd_term(s.float(), t.float(), T)[m].mean().clamp(max=cfg.scrub_kl_cap))
        for x, y, r, _ in loader(cfg, imgs, [given, role], idx, shuffle=True, augment=True, seed=seed + 50 + ep):
            x, y, r = x.to(cfg.device), y.to(cfg.device), r.to(cfg.device)
            with torch.no_grad(), autocast(cfg):
                t = teacher(x)
            with autocast(cfg):
                s = student(x).float()
            m = (r == 0) & (y != IGNORE)
            if m.any():
                step(kd_term(s, t.float(), T)[m].mean() + F.cross_entropy(
                    s, torch.where(m, y, torch.full_like(y, IGNORE)), ignore_index=IGNORE))
    return student.eval()


def run_ssd(cfg, f0, imgs, given, forget, seed):
    model = full_copy(f0)
    role = forget.astype(np.uint8) * 2
    rng = np.random.default_rng(seed)
    fidx = np.nonzero(role.reshape(len(role), -1).max(1) > 0)[0]
    didx = rng.choice(len(imgs), min(len(imgs), max(len(fidx), 1)), replace=False)

    def fisher(idx, forget_only):
        acc = {n: torch.zeros_like(p) for n, p in model.named_parameters()}
        nb = 0
        for x, y, r, _ in loader(cfg, imgs, [given, role], idx, seed=seed):
            x, y, r = x.to(cfg.device), y.to(cfg.device), r.to(cfg.device)
            yy = torch.where(r > 0, y, torch.full_like(y, IGNORE)) if forget_only else y
            if not (yy != IGNORE).any():
                continue
            model.zero_grad(set_to_none=True)
            F.cross_entropy(model(x).float(), yy, ignore_index=IGNORE).backward()
            for n, p in model.named_parameters():
                if p.grad is not None:
                    acc[n] += p.grad.detach() ** 2
            nb += 1
        return {n: v / max(nb, 1) for n, v in acc.items()}

    model.eval()
    I_f = fisher(fidx, True)
    I_d = fisher(didx, False)
    with torch.no_grad():
        for n, p in model.named_parameters():
            sel = I_f[n] > cfg.ssd_alpha * I_d[n]
            beta = torch.clamp(cfg.ssd_lambda * I_d[n] / (I_f[n] + 1e-12), max=1.0)
            p[sel] = p[sel] * beta[sel]
    model.zero_grad(set_to_none=True)
    return model.eval()


BASELINES = {'finetune': run_finetune, 'neggrad': run_neggrad, 'scrub': run_scrub, 'ssd': run_ssd}


# --------------------------------------------------------------------------- figures
PALETTE = np.array([[0, 0, 0], [255, 0, 0], [180, 120, 120], [160, 150, 20], [140, 140, 140],
                    [61, 230, 250], [0, 82, 255], [255, 0, 245], [255, 235, 0], [4, 250, 7]], dtype=np.uint8)


def colorize(m):
    out = PALETTE[np.minimum(m, K - 1)]
    out[m == IGNORE] = 255
    return out


def _plt():
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    return plt


def fig_dir(cfg):
    d = os.path.join(cfg.work_dir, 'results', 'figures')
    os.makedirs(d, exist_ok=True)
    return d


def plot_curves(cfg):
    plt = _plt()
    hists = sorted(glob.glob(os.path.join(cfg.work_dir, 'models', '*', 'history.csv')))
    if not hists:
        return
    plt.figure(figsize=(8, 4.5))
    for h in hists:
        df = pd.read_csv(h)
        if 'val_miou' in df:
            d = df.dropna(subset=['val_miou'])
            plt.plot(d['epoch'], d['val_miou'], marker='o', label=df['tag'].iloc[0])
    plt.xlabel('epoch')
    plt.ylabel('validation mIoU')
    plt.legend(fontsize=7)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir(cfg), 'training_curves.png'), dpi=150)
    plt.close()


def plot_examples(cfg, imgs, clean, noisy, role, before, after):
    plt = _plt()
    fig, ax = plt.subplots(len(imgs), 6, figsize=(18, 2.6 * len(imgs)))
    ax = np.atleast_2d(ax)
    titles = ['image', 'original label', 'corrupted label', 'flags (blue correct, orange forget)',
              'before CRU-Flood', 'after CRU-Flood']
    for r in range(len(imgs)):
        flag = np.zeros(imgs[r].shape, dtype=np.uint8) + 230
        flag[role[r] == 1] = [40, 90, 220]
        flag[role[r] == 2] = [240, 140, 30]
        for c, p in enumerate([imgs[r], colorize(clean[r]), colorize(noisy[r]), flag,
                               colorize(before[r]), colorize(after[r])]):
            ax[r, c].imshow(p)
            ax[r, c].axis('off')
            if r == 0:
                ax[r, c].set_title(titles[c], fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir(cfg), 'selection_examples.png'), dpi=130)
    plt.close()


def plot_real(cfg, crops, imgs, given, ii, preds):
    plt = _plt()
    fig, ax = plt.subplots(len(crops), 3, figsize=(9, 2.8 * len(crops)))
    ax = np.atleast_2d(ax)
    for r, (i, y0, y1, x0, x1) in enumerate(crops):
        for c, p in enumerate([imgs[i], colorize(given[i]), colorize(preds[r])]):
            ax[r, c].imshow(p[y0:y1, x0:x1])
            ax[r, c].axis('off')
            if r == 0:
                ax[r, c].set_title(['image', 'given label', 'model prediction'][c], fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir(cfg), 'real_label_flags.png'), dpi=130)
    plt.close()


# --------------------------------------------------------------------------- experiment driver
TRAIN_FIELDS = ('width', 'height', 'models', 'encoders', 'pretrained', 'epochs', 'early_frac', 'batch_size', 'lr',
                'mc_p', 'min_area', 'noise_rates', 'amp', 'smoke')
NOT_SETTINGS = ('data_root', 'work_dir', 'resume_from', 'stage', 'parts', 'device', 'num_workers')


def fingerprint(cfg, fields=None):
    d = {k: v for k, v in asdict(cfg).items() if (k in fields if fields else k not in NOT_SETTINGS)}
    return hashlib.sha1((RUN_VERSION + json.dumps(d, sort_keys=True, default=str)).encode()).hexdigest()[:12]


def check_fresh(cfg):
    """Move aside earlier output made with other code or settings, so results from different runs never mix."""
    fp_path = os.path.join(cfg.work_dir, 'run_fingerprint.json')
    new = {'settings': fingerprint(cfg), 'training': fingerprint(cfg, TRAIN_FIELDS)}
    old = json.load(open(fp_path)) if os.path.exists(fp_path) else None
    present = [d for d in ('models', 'cache', 'results') if os.path.isdir(os.path.join(cfg.work_dir, d))]
    if present and old != new:
        keep_models = old is not None and old.get('training') == new['training']
        moved = ['results'] if keep_models else ['results', 'models', 'cache']
        dest = os.path.join(cfg.work_dir, 'old_' + time.strftime('%Y%m%d-%H%M%S'))
        os.makedirs(dest, exist_ok=True)
        for d in moved:
            if os.path.isdir(os.path.join(cfg.work_dir, d)):
                shutil.move(os.path.join(cfg.work_dir, d), os.path.join(dest, d))
        if keep_models:
            for f in glob.glob(os.path.join(cfg.work_dir, 'cache', 'forget_*.npy')):
                shutil.move(f, os.path.join(dest, os.path.basename(f)))
        log(f'Earlier output in {cfg.work_dir} came from other code or settings; moved {", ".join(moved)} to {dest}. '
            + ('Base models are reused.' if keep_models else 'Everything is recomputed.'))
    json.dump(new, open(fp_path, 'w'))


def smoke_banner(cfg):
    if cfg.smoke:
        log('*' * 72)
        log('SMOKE TEST: tiny settings that only check the code runs. These numbers are NOT results.')
        log('For the article, set SMOKE_TEST = False and run again.')
        log('*' * 72)


def run(cfg):
    os.makedirs(cfg.work_dir, exist_ok=True)
    if cfg.resume_from and os.path.isdir(cfg.resume_from):
        for sub in ('models', 'cache', 'results'):
            src = os.path.join(cfg.resume_from, sub)
            if os.path.isdir(src):
                shutil.copytree(src, os.path.join(cfg.work_dir, sub), dirs_exist_ok=True)
        fp_src = os.path.join(cfg.resume_from, 'run_fingerprint.json')
        if os.path.exists(fp_src):
            shutil.copy(fp_src, os.path.join(cfg.work_dir, 'run_fingerprint.json'))
    smoke_banner(cfg)
    check_fresh(cfg)
    res = os.path.join(cfg.work_dir, 'results')
    os.makedirs(res, exist_ok=True)
    env = {'run_version': RUN_VERSION, 'smoke_test': cfg.smoke, 'python': platform.python_version(), 'torch': torch.__version__, 'smp': smp.__version__,
           'device': cfg.device, 'gpu': torch.cuda.get_device_name(0) if cfg.device == 'cuda' else None,
           'config': {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(cfg).items()}}
    json.dump(env, open(os.path.join(res, 'environment.json'), 'w'), indent=2)
    done = Done(os.path.join(res, 'done.json'))
    data = load_floodnet(cfg)
    env['fallback_split'] = data.pop('fallback_split')
    json.dump(env, open(os.path.join(res, 'environment.json'), 'w'), indent=2)
    tr = data['train']
    clean = tr['masks']

    # ---- Stage 1: base models
    paths, corr, loaded = {}, {}, {}

    def get_corr(eta):
        if eta not in loaded:
            loaded.clear()
            z = np.load(corr[eta])
            loaded[eta] = (z['noisy'], z['comp_map'] > 0, z['comp_map'])
        return loaded[eta]
    for name in cfg.models:
        paths[(name, 'clean')] = train_model(cfg, name, data, clean, f'{name}_clean')
    for eta in cfg.noise_rates:
        cpath = os.path.join(cfg.work_dir, 'cache', f'corrupt_eta{eta}.npz')
        if os.path.exists(cpath):
            z = np.load(cpath)
            noisy, stats = z['noisy'], json.loads(str(z['stats']))
        else:
            noisy, C, comp_map, stats = corrupt(clean, eta, seed=1000 + int(round(eta * 100)), min_area=cfg.min_area)
            np.savez_compressed(cpath, noisy=noisy, comp_map=comp_map, stats=json.dumps(stats))
            del C, comp_map
        corr[eta] = cpath
        if f'corrupt_{eta}' not in done:
            append_row(os.path.join(res, 'corruption_stats.csv'), stats)
            done.add(f'corrupt_{eta}')
        for name in cfg.models:
            paths[(name, eta)] = train_model(cfg, name, data, noisy, f'{name}_eta{eta}')
    for (name, setting), (fp, _) in paths.items():
        key = f'base_{name}_{setting}'
        if key in done:
            continue
        m = load_model(cfg, name, fp)
        row = {'model': name, 'setting': setting, **evaluate(cfg, m, data['test']['imgs'], data['test']['masks'])}
        meta = os.path.join(os.path.dirname(fp), 'meta.json')
        if os.path.exists(meta):
            row['train_seconds'] = json.load(open(meta))['train_seconds']
        if setting != 'clean':
            noisy_s, C_s, _ = get_corr(setting)
            row.update(train_set_metrics(cfg, m, tr['imgs'], clean, noisy_s, C_s))
        append_row(os.path.join(res, 'base_models.csv'), row)
        done.add(key)
    if cfg.stage == 'train':
        log('Stage 1 finished')
        return

    # ---- Stage 2: unlearning
    segs = compute_slic(cfg, tr['imgs'])
    test = data['test']
    peer_of = {cfg.models[0]: cfg.models[-1], cfg.models[-1]: cfg.models[0]}
    sig_cache, peer_cache = {}, {}

    def models_for(name, setting):
        fp, ep = paths[(name, setting)]
        return load_model(cfg, name, fp), load_model(cfg, name, ep)

    def peer_labels(name, setting):
        key = (name, setting)
        if key not in peer_cache:
            peer = peer_of[name]
            _, g_early = models_for(peer, setting)
            peer_cache[key] = predict_labels(cfg, g_early, tr['imgs'])
        return peer_cache[key]

    def signals(name, setting, given):
        key = (name, setting)
        if key not in sig_cache:
            _, f_early = models_for(name, setting)
            sig_cache[key] = compute_signals(cfg, f_early, tr['imgs'], given, seed=0)
        return sig_cache[key]

    def result_row(model, name, setting, method, seed, given_noisy=None, C=None, **kw):
        row = {'model': name, 'setting': setting, 'method': method, 'seed': seed,
               **evaluate(cfg, model, test['imgs'], test['masks']), **kw}
        if C is not None:
            row.update(train_set_metrics(cfg, model, tr['imgs'], clean, given_noisy, C))
        return row

    ucsv = os.path.join(res, 'unlearning.csv')

    def forget_set(name, eta, noisy, sig1, pl):
        fpath = os.path.join(cfg.work_dir, 'cache', f'forget_{name}_eta{eta}.npy')
        if not os.path.exists(fpath):
            role, _, _ = select_regions(noisy, segs, sig1, pl, cfg.weights, cfg.tau)
            np.save(fpath, role > 0)
        return np.load(fpath)

    tuned_path = os.path.join(res, 'tuned.json')
    tuned = json.load(open(tuned_path)) if os.path.exists(tuned_path) else {}

    def settings(name, method):
        params = dict(tuned.get(name, {}).get(method, {}))
        if method == 'cru_flood' and 'uncertain_mode' not in params:
            params['uncertain_mode'] = 'forget' if cfg.uncertain_mode == 'auto' else cfg.uncertain_mode
        return dataclasses.replace(cfg, **params)

    if any(p in cfg.parts for p in ('main', 'ablations', 'idfrac', 'real')):
        eta = cfg.main_eta
        noisy, C, _ = get_corr(eta)
        for name in cfg.models:
            if name in tuned:
                continue
            f_final, f_early = models_for(name, eta)
            pl = peer_labels(name, eta)
            sig1 = signals(name, eta, noisy)
            forget = forget_set(name, eta, noisy, sig1, pl)
            choice = {}
            grid = cfg.tune_grid if cfg.tune else {}
            if cfg.uncertain_mode != 'auto':
                grid = {m: g for m, g in grid.items() if m != 'cru_flood'}
            for method, options in grid.items():
                best = None
                for params in options:
                    c2 = dataclasses.replace(cfg, **params)
                    seed = cfg.seeds[0]
                    log(f'Tuning {method} {name} {params}')
                    if method == 'cru_flood':
                        model, _ = run_cru(c2, f_final, f_early, pl, tr['imgs'], noisy, segs, seed, C=C, sig1=sig1)
                    else:
                        model = BASELINES[method](c2, f_final, tr['imgs'], noisy, forget, seed)
                    val = evaluate(cfg, model, data['val']['imgs'], data['val']['masks'])
                    append_row(os.path.join(res, 'tuning.csv'), {'model': name, 'method': method, 'params': json.dumps(params),
                                                                 'val_miou': val['miou'], 'val_pair_miou': val['pair_miou']})
                    score = (round(val['miou'], 6), round(val['pair_miou'], 6))
                    if best is None or score > best[0]:
                        best = (score, params)
                choice[method] = best[1]
                log(f'  chosen for {method}: {best[1]}')
            tuned[name] = choice
            json.dump(tuned, open(tuned_path, 'w'), indent=2)

    if 'main' in cfg.parts:
        for eta in cfg.noise_rates:
            noisy, C, comp_map = get_corr(eta)
            for name in cfg.models:
                f_final, f_early = models_for(name, eta)
                pl = peer_labels(name, eta)
                sig1 = signals(name, eta, noisy)
                if f'orig_{name}_{eta}' not in done:
                    append_row(ucsv, result_row(f_final, name, eta, 'original', -1, noisy, C))
                    oracle, _ = models_for(name, 'clean')
                    append_row(ucsv, result_row(oracle, name, eta, 'clean_retrain', -1, noisy, C))
                    done.add(f'orig_{name}_{eta}')
                seeds = cfg.seeds if eta == cfg.main_eta else cfg.seeds[:1]
                for seed in seeds:
                    key = f'cru_{name}_{eta}_{seed}'
                    if key not in done:
                        log(f'CRU-Flood {name} eta={eta} seed={seed}')
                        model, info = run_cru(settings(name, 'cru_flood'), f_final, f_early, pl, tr['imgs'], noisy, segs,
                                              seed, C=C, sig1=sig1)
                        append_row(ucsv, result_row(model, name, eta, 'cru_flood', seed, noisy, C,
                                                    seconds=info['seconds'], trainable_params=info['trainable']))
                        for lg in info['logs']:
                            append_row(os.path.join(res, 'selection_rounds.csv'),
                                       {'model': name, 'setting': eta, 'seed': seed, **lg})
                        if eta == cfg.main_eta and name == cfg.ablation_model and seed == seeds[0]:
                            nC = C.reshape(len(C), -1).sum(1)
                            ex_idx = np.argsort(-nC)[:4]
                            plot_examples(cfg, tr['imgs'][ex_idx], clean[ex_idx], noisy[ex_idx], info['role'][ex_idx],
                                          predict_labels(cfg, f_final, tr['imgs'][ex_idx]),
                                          predict_labels(cfg, model, tr['imgs'][ex_idx]))
                        done.add(key)
                    forget = forget_set(name, eta, noisy, sig1, pl)
                    for method, fn in BASELINES.items():
                        key = f'{method}_{name}_{eta}_{seed}'
                        if key in done:
                            continue
                        log(f'{method} {name} eta={eta} seed={seed}')
                        set_seed(seed)
                        t0 = time.time()
                        model = fn(settings(name, method), f_final, tr['imgs'], noisy, forget, seed)
                        append_row(ucsv, result_row(model, name, eta, method, seed, noisy, C,
                                                    seconds=time.time() - t0,
                                                    trainable_params=0 if method == 'ssd' else n_trainable(model)))
                        done.add(key)
                if cfg.run_rewod and eta == cfg.main_eta and f'rewod_{name}_{eta}' not in done:
                    forget = forget_set(name, eta, noisy, sig1, pl)
                    masked = noisy.copy()
                    masked[forget] = IGNORE
                    t0 = time.time()
                    fp, _ = train_model(cfg, name, data, masked, f'{name}_eta{eta}_rewod')
                    model = load_model(cfg, name, fp)
                    append_row(ucsv, result_row(model, name, eta, 'retrain_without_forget', -1, noisy, C,
                                                seconds=time.time() - t0, trainable_params=sum(p.numel() for p in model.parameters())))
                    done.add(f'rewod_{name}_{eta}')
            if eta != cfg.main_eta:
                for d in (sig_cache, peer_cache):
                    for key in [k for k in d if k[1] == eta]:
                        del d[key]

    eta = cfg.main_eta
    noisy, C, comp_map = get_corr(eta)
    if 'tau' in cfg.parts:
        for name in cfg.models:
            pl = peer_labels(name, eta)
            sig = signals(name, eta, noisy)
            for tau in cfg.tau_grid:
                key = f'tau_{name}_{tau}'
                if key in done:
                    continue
                role, _, info = select_regions(noisy, segs, sig, pl, cfg.weights, tau)
                append_row(os.path.join(res, 'tau_sensitivity.csv'),
                           {'model': name, 'setting': eta, 'tau': tau, **info, **selection_stats(role, C, noisy)})
                done.add(key)

    if 'ablations' in cfg.parts:
        name = cfg.ablation_model
        f_final, f_early = models_for(name, eta)
        pl = peer_labels(name, eta)
        sig = signals(name, eta, noisy)
        base_role, _, base_info = select_regions(noisy, segs, sig, pl, cfg.weights, cfg.tau)
        budget = int((base_role > 0).sum())
        sel_variants = {
            'all three signals': dict(w=cfg.weights),
            'loss rank only': dict(w=(1, 0, 0)), 'uncertainty only': dict(w=(0, 1, 0)),
            'agreement only': dict(w=(0, 0, 1)), 'without loss rank': dict(w=(0, 0.5, 0.5)),
            'without uncertainty': dict(w=(0.5, 0, 0.5)), 'without agreement': dict(w=(0.5, 0.5, 0)),
            'pixel level': dict(w=cfg.weights, pixel_level=True)}
        for vname, kw in sel_variants.items():
            key = f'abl_sel_{vname}'
            if key in done:
                continue
            role, _, info = select_regions(noisy, segs, sig, pl, kw['w'], cfg.tau, budget=budget,
                                           pixel_level=kw.get('pixel_level', False))
            append_row(os.path.join(res, 'ablation_selection.csv'),
                       {'model': name, 'variant': vname, 'budget_px': budget, **info, **selection_stats(role, C, noisy)})
            done.add(key)
        key = 'abl_sel_final snapshot'
        if key not in done:
            sig_f = compute_signals(cfg, f_final, tr['imgs'], noisy, seed=0)
            pl_f = predict_labels(cfg, models_for(peer_of[name], eta)[0], tr['imgs'])
            role, _, info = select_regions(noisy, segs, sig_f, pl_f, cfg.weights, cfg.tau, budget=budget)
            append_row(os.path.join(res, 'ablation_selection.csv'),
                       {'model': name, 'variant': 'final weights instead of early snapshot', 'budget_px': budget,
                        **info, **selection_stats(role, C, noisy)})
            done.add(key)
        cru_cfg = settings(name, 'cru_flood')
        unl_variants = {
            'CRU-Flood (full)': dict(),
            'one round': dict(rounds=1),
            'without relabelling': dict(forget_only=True),
            'unagreed regions forgotten': dict(mode='forget'),
            'unagreed regions ignored': dict(mode='ignore'),
            'without distillation': dict(lam_kd=0.0),
            'full fine-tuning instead of LoRA': dict(full_ft=True)}
        for vname, kw in unl_variants.items():
            kw = dict(kw)
            vcfg = dataclasses.replace(cru_cfg, uncertain_mode=kw.pop('mode')) if 'mode' in kw else cru_cfg
            for seed in cfg.seeds[:1]:
                key = f'abl_unl_{vname}_{seed}'
                if key in done:
                    continue
                log(f'Ablation: {vname}')
                model, info = run_cru(vcfg, f_final, f_early, pl, tr['imgs'], noisy, segs, seed, C=C, sig1=sig, **kw)
                append_row(os.path.join(res, 'ablation_unlearning.csv'),
                           {**result_row(model, name, eta, vname, seed, noisy, C), 'uncertain_mode': vcfg.uncertain_mode,
                            'seconds': info['seconds'],
                            'trainable_params': info['trainable'], 'rounds_run': len(info['logs'])})
                done.add(key)

    if 'idfrac' in cfg.parts:
        name = cfg.ablation_model
        f_final, f_early = models_for(name, eta)
        n_comp = int(comp_map.max())
        for frac in cfg.id_fractions:
            rng = np.random.default_rng(int(frac * 1000))
            ids = rng.choice(np.arange(1, n_comp + 1), max(1, int(round(frac * n_comp))), replace=False)
            known = np.isin(comp_map, ids)
            for method in ('cru_flood_oracle', 'finetune', 'scrub'):
                for seed in cfg.seeds[:1]:
                    key = f'idfrac_{frac}_{method}_{seed}'
                    if key in done:
                        continue
                    log(f'Identification fraction {frac}: {method}')
                    t0 = time.time()
                    if method == 'cru_flood_oracle':
                        model = copy.deepcopy(f_final)
                        apply_lora(model, cfg.lora_r, cfg.lora_alpha)
                        role = known.astype(np.uint8)
                        target = np.where(known, PARTNER[noisy], IGNORE).astype(np.uint8)
                        train_cru(cfg, model, tr['imgs'], noisy, role, target, seed)
                        merge_lora(model)
                    else:
                        model = BASELINES[method](settings(name, method), f_final, tr['imgs'], noisy, known, seed)
                    append_row(os.path.join(res, 'identification_fraction.csv'),
                               {**result_row(model, name, eta, method, seed, noisy, C),
                                'identified_fraction': frac, 'seconds': time.time() - t0})
                    done.add(key)

    if 'real' in cfg.parts:
        sig_cache.clear()
        for name in cfg.models:
            key = f'real_{name}'
            if key in done:
                continue
            log(f'Real-label setting: {name}')
            f_final, f_early = models_for(name, 'clean')
            pl = peer_labels(name, 'clean')
            model, info = run_cru(settings(name, 'cru_flood'), f_final, f_early, pl, tr['imgs'], clean, segs, seed=0)
            before = evaluate(cfg, f_final, test['imgs'], test['masks'])
            after = evaluate(cfg, model, test['imgs'], test['masks'])
            row = {'model': name, 'flagged_px': int((info['role'] > 0).sum()),
                   'pct_flood_pair_px': 100 * (info['role'] > 0).sum() / max(IS_PAIR[clean].sum(), 1),
                   'seconds': info['seconds']}
            row.update({f'before_{k}': v for k, v in before.items()})
            row.update({f'after_{k}': v for k, v in after.items()})
            append_row(os.path.join(res, 'real_labels.csv'), row)
            if name == cfg.ablation_model:
                role = info['role']
                crops = []
                struct = np.zeros((3, 3, 3), dtype=bool)
                struct[1] = True                      # connect within an image, never across images
                lab, k = ndimage.label(role > 0, structure=struct)
                if k:
                    sizes = np.bincount(lab.ravel())[1:]
                    for lid in np.argsort(-sizes)[:8] + 1:
                        sl = ndimage.find_objects((lab == lid).astype(np.int8))[0]
                        i = sl[0].start
                        y0, y1 = max(0, sl[1].start - 24), min(cfg.height, sl[1].stop + 24)
                        x0, x1 = max(0, sl[2].start - 24), min(cfg.width, sl[2].stop + 24)
                        crops.append((i, y0, y1, x0, x1))
                if crops:
                    ii = np.array([c[0] for c in crops])
                    plot_real(cfg, crops, tr['imgs'], clean, ii, predict_labels(cfg, f_final, tr['imgs'][ii]))
            done.add(key)

    plot_curves(cfg)
    zname = 'SMOKE_TEST_not_results' if cfg.smoke else 'cru_flood_results'
    shutil.make_archive(os.path.join(cfg.work_dir, zname), 'zip', res)
    log(f'Finished. Results: {res} and {cfg.work_dir}/{zname}.zip')
    smoke_banner(cfg)


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--data_root', default='')
    ap.add_argument('--work_dir', default='./cru_flood_run')
    ap.add_argument('--tiny', action='store_true', help='CPU smoke test settings')
    ap.add_argument('--stage', default='all')
    a = ap.parse_args()
    cfg = tiny_config() if a.tiny else Config()
    cfg.data_root, cfg.work_dir, cfg.stage = a.data_root, a.work_dir, a.stage
    run(cfg)


## Settings

In [ ]:
import importlib, cruflood
importlib.reload(cruflood)

SMOKE_TEST = False          # False: the real experiments for the article. True: a few-minute code check (not results).
STAGE = 'all'               # 'train', 'unlearn' or 'all'
PARTS = ('main', 'tau', 'ablations', 'idfrac', 'real')
RESUME_FROM = ''            # e.g. '/kaggle/input/<your-session-1-output>/cru_flood'
DATA_ROOT = ''              # leave empty to search /kaggle/input automatically

if SMOKE_TEST:
    import shutil
    shutil.rmtree('/kaggle/working/smoke_test', ignore_errors=True)   # every quick check starts fresh
    cfg = cruflood.tiny_config(work_dir='/kaggle/working/smoke_test')
else:
    cfg = cruflood.Config(work_dir='/kaggle/working/cru_flood')
cfg.stage, cfg.parts, cfg.resume_from, cfg.data_root = STAGE, PARTS, RESUME_FROM, DATA_ROOT
print(cfg)

## Check the dataset
Lists the folders with images under the input folder and counts the image/mask pairs found for each split. Expect 238 / 79 / 81 for the FloodNet Challenge copy (or about 1,445 / 450 / 448 for FloodNet-Supervised_v1.0). If this cell fails, copy its full output into the chat.

In [ ]:
print(cruflood.describe_inputs(DATA_ROOT or '/kaggle/input'))
pairs = cruflood.find_floodnet(DATA_ROOT)
print({k: len(v) for k, v in pairs.items() if k != 'fallback_split'})

## Run

In [ ]:
cruflood.run(cfg)

## Summary of the results files

In [ ]:
import os, pandas as pd
if cfg.smoke:
    print('SMOKE TEST: these tables only show that the code runs. They are NOT results for the article.')
res = os.path.join(cfg.work_dir, 'results')
for f in sorted(os.listdir(res)):
    if f.endswith('.csv'):
        print('\n==', f)
        df = pd.read_csv(os.path.join(res, f))
        if f == 'unlearning.csv':
            cols = [c for c in ['miou', 'pair_miou', 'correction_rate', 'memorisation_rate', 'retain_acc', 'ece', 'seconds'] if c in df]
            display(df.groupby(['model', 'setting', 'method'])[cols].agg(['mean', 'std']).round(4))
        else:
            display(df.round(4))